In [1]:
# Performance config
import os

CPU_THREADS = min(32, os.cpu_count() or 32)
os.environ["OMP_NUM_THREADS"] = str(CPU_THREADS)
os.environ["MKL_NUM_THREADS"] = str(CPU_THREADS)
os.environ["OPENBLAS_NUM_THREADS"] = str(CPU_THREADS)
os.environ["NUMEXPR_NUM_THREADS"] = str(CPU_THREADS)
os.environ["VECLIB_MAXIMUM_THREADS"] = str(CPU_THREADS)
os.environ["TOKENIZERS_PARALLELISM"] = "true"

REQUIRE_CUDA = True  # set False for CPU-only notebooks

print(f"CPU threads set to: {CPU_THREADS}")

try:
    import torch
except Exception as e:
    torch = None
    if REQUIRE_CUDA:
        raise RuntimeError("CUDA required but torch is not available.") from e

if torch is not None:
    torch.set_num_threads(CPU_THREADS)
    torch.set_num_interop_threads(min(4, CPU_THREADS))
    if REQUIRE_CUDA and not torch.cuda.is_available():
        raise RuntimeError("CUDA required but not available.")
    if torch.cuda.is_available():
        torch.backends.cuda.matmul.allow_tf32 = True
        print("CUDA device:", torch.cuda.get_device_name(0))
    else:
        print("CUDA not available; running on CPU.")


CPU threads set to: 32
CUDA device: NVIDIA B200


# Build Kvasir-VQA x1 metadata + image manifest

Prep pipeline for the Kvasir-VQA x1 dataset: download images, write manifests/metadata, and record basic stats.

In [2]:
import os
from pathlib import Path
from urllib.parse import urlparse
import random

import numpy as np
import pandas as pd
from PIL import Image as PILImage
from tqdm.auto import tqdm

try:
    import torch
    TORCH_AVAILABLE = True
except Exception as e:
    TORCH_AVAILABLE = False
    torch = None

from datasets import load_dataset, get_dataset_infos, Image as HFImage


In [3]:
# Config
HF_DATASET_ID = "SimulaMet/Kvasir-VQA-x1"
HOST_IMAGE_DATASET_ID = "SimulaMet-HOST/Kvasir-VQA"
HOST_IMAGE_SPLIT = "raw"
SPLITS = ["train", "validation", "test"]  # if dataset has single split, will fall back to that
MAX_SAMPLES_PER_SPLIT = int(os.getenv("MAX_SAMPLES_PER_SPLIT", "0")) or None  # set to int/env for smoke test

# Threading / workers
try:
    CPU_THREADS
except NameError:
    CPU_THREADS = min(32, os.cpu_count() or 32)
NUM_WORKERS = int(os.getenv("NUM_WORKERS", "0")) or CPU_THREADS
NUM_WORKERS = max(1, min(NUM_WORKERS, os.cpu_count() or NUM_WORKERS))

# GPU config (optional image round-trip; disabled by default for speed)
FORCE_CPU = os.getenv("FORCE_CPU", "0") == "1"
GPU_IMAGE_PIPELINE = os.getenv("GPU_IMAGE_PIPELINE", "0") == "1"
USE_GPU = bool(TORCH_AVAILABLE and (not FORCE_CPU) and torch.cuda.is_available())
GPU_IMAGE_PIPELINE = bool(GPU_IMAGE_PIPELINE and USE_GPU)
DEVICE = torch.device("cuda" if USE_GPU else "cpu") if TORCH_AVAILABLE else "cpu"

SPLIT_SEED = 42
TRAIN_FRAC = 0.8
VAL_FRAC = 0.1
SPLIT_BY_IMAGE = True  # avoid leakage across questions for same image

OUT_ROOT = Path("./out")
IMAGES_DIR = OUT_ROOT / "images"
HOST_IMAGE_DIR = IMAGES_DIR / "all"
META_DIR = OUT_ROOT / "metadata"
MANIFEST_DIR = OUT_ROOT / "manifests"
for d in [IMAGES_DIR, HOST_IMAGE_DIR, META_DIR, MANIFEST_DIR]:
    d.mkdir(parents=True, exist_ok=True)

RAW_META_CSV = META_DIR / "metadata_raw.csv"
ENRICHED_META_CSV = META_DIR / "metadata_enriched.csv"
IMAGE_MANIFEST_CSV = MANIFEST_DIR / "image_manifest.csv"

# Download dataset snapshot locally to avoid HF rate limits
USE_LOCAL_SNAPSHOT = os.getenv("USE_LOCAL_SNAPSHOT", "1") == "1"
# To disable local snapshot behavior, set: USE_LOCAL_SNAPSHOT=0
FORCE_SNAPSHOT_DOWNLOAD = os.getenv("FORCE_SNAPSHOT_DOWNLOAD", "0") == "1"
SNAPSHOT_MAX_WORKERS = int(os.getenv("SNAPSHOT_MAX_WORKERS", "2"))  # lower to reduce rate-limit risk
SNAPSHOT_RESUME = os.getenv("SNAPSHOT_RESUME", "1") == "1"
SNAPSHOT_ALLOW_PATTERNS = ["data/**", "README.md"]
LOCAL_SNAPSHOT_DIR = OUT_ROOT / "hf_snapshot" / "kvasir-vqa-x1"

# Host image download (recommended, per dataset README)
USE_HOST_IMAGES = os.getenv("USE_HOST_IMAGES", "1") == "1"
HOST_SNAPSHOT_DIR = OUT_ROOT / "hf_snapshot" / "kvasir-vqa-host"
HOST_SNAPSHOT_MAX_WORKERS = int(os.getenv("HOST_SNAPSHOT_MAX_WORKERS", "2"))
HOST_SNAPSHOT_RESUME = os.getenv("HOST_SNAPSHOT_RESUME", "1") == "1"
HOST_SNAPSHOT_ALLOW_PATTERNS = ["data/**", "README.md"]
READ_IMAGE_SIZES = os.getenv("READ_IMAGE_SIZES", "0") == "1"

# Show a single overall progress bar (disable per-file bars)
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"


def _local_snapshot_ready(path: Path) -> bool:
    return path.exists() and any(path.iterdir())


def _maybe_set_offline():
    os.environ["HF_DATASETS_OFFLINE"] = "1"
    os.environ["HF_HUB_OFFLINE"] = "1"


def _ensure_host_images():
    if not USE_HOST_IMAGES:
        return
    if HOST_IMAGE_DIR.exists() and any(HOST_IMAGE_DIR.iterdir()):
        return

    from huggingface_hub import snapshot_download
    from tqdm.auto import tqdm

    HOST_SNAPSHOT_DIR.mkdir(parents=True, exist_ok=True)
    # Download host dataset snapshot (single progress bar)
    snapshot_download(
        repo_id=HOST_IMAGE_DATASET_ID,
        repo_type="dataset",
        local_dir=str(HOST_SNAPSHOT_DIR),
        max_workers=HOST_SNAPSHOT_MAX_WORKERS,
        resume_download=HOST_SNAPSHOT_RESUME,
        allow_patterns=HOST_SNAPSHOT_ALLOW_PATTERNS,
        tqdm_class=tqdm,
    )

    # Load host dataset locally and save unique images
    ds_host = load_dataset(str(HOST_SNAPSHOT_DIR), split=HOST_IMAGE_SPLIT)
    seen = set()
    for row in tqdm(ds_host, desc="Saving host images"):
        img_id = row.get("img_id")
        if not img_id or img_id in seen:
            continue
        out_path = HOST_IMAGE_DIR / f"{img_id}.jpg"
        if not out_path.exists():
            row["image"].save(out_path)
        seen.add(img_id)


# Prepare QA dataset (x1)
if USE_LOCAL_SNAPSHOT:
    from huggingface_hub import snapshot_download
    from tqdm.auto import tqdm as _tqdm

    LOCAL_SNAPSHOT_DIR.mkdir(parents=True, exist_ok=True)
    if _local_snapshot_ready(LOCAL_SNAPSHOT_DIR) and not FORCE_SNAPSHOT_DOWNLOAD:
        HF_DATASET = str(LOCAL_SNAPSHOT_DIR)
        print("Using existing local snapshot:", HF_DATASET)
    else:
        HF_DATASET = snapshot_download(
            repo_id=HF_DATASET_ID,
            repo_type="dataset",
            local_dir=str(LOCAL_SNAPSHOT_DIR),
            max_workers=SNAPSHOT_MAX_WORKERS,
            resume_download=SNAPSHOT_RESUME,
            allow_patterns=SNAPSHOT_ALLOW_PATTERNS,
            tqdm_class=_tqdm,
        )
else:
    HF_DATASET = HF_DATASET_ID

print("Dataset:", HF_DATASET)
print("Output root:", OUT_ROOT)
print("Workers:", NUM_WORKERS)
if TORCH_AVAILABLE:
    print("GPU enabled:", USE_GPU, "| GPU image pipeline:", GPU_IMAGE_PIPELINE)
else:
    print("Torch not available; running CPU-only.")


Using existing local snapshot: out/hf_snapshot/kvasir-vqa-x1
Dataset: out/hf_snapshot/kvasir-vqa-x1
Output root: out
Workers: 32
GPU enabled: True | GPU image pipeline: False


In [4]:
def safe_img_id(ex, split: str, idx: int) -> str:
    # Try common keys for image filename
    for k in ["image_id", "img_id", "id", "filename"]:
        if k in ex and ex[k]:
            return str(ex[k]).split("/")[-1].split(".")[0]
    return f"{split}_{idx:06d}"


def hf_url_to_local_path(url: str, snapshot_dir: Path):
    if not url.startswith("https://huggingface.co/datasets/"):
        return None
    try:
        path = urlparse(url).path  # /datasets/<repo>/resolve/<rev>/...
        if "/resolve/" not in path:
            return None
        rel = path.split("/resolve/", 1)[1]
        # drop the revision segment
        if "/" in rel:
            rel = rel.split("/", 1)[1]
        return snapshot_dir / rel
    except Exception:
        return None


def maybe_gpu_roundtrip(img):
    if not GPU_IMAGE_PIPELINE:
        return img
    if img is None:
        return img
    if hasattr(img, "mode") and img.mode != "RGB":
        img = img.convert("RGB")
    arr = np.array(img)
    if not arr.flags["C_CONTIGUOUS"]:
        arr = np.ascontiguousarray(arr)
    tensor = torch.from_numpy(arr).to(DEVICE)
    tensor = tensor.contiguous()
    arr_out = tensor.to("cpu").numpy()
    return PILImage.fromarray(arr_out)


def save_image(img, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    img = maybe_gpu_roundtrip(img)
    if hasattr(img, "mode") and img.mode != "RGB":
        img = img.convert("RGB")
    img.save(path, format="JPEG")


def ensure_image_column(ds):
    # If using host images or local snapshot, avoid decoding image URLs here.
    if USE_HOST_IMAGES or USE_LOCAL_SNAPSHOT:
        return ds
    # Cast string/bytes image column to decoded PIL images when needed
    if "image" in ds.column_names and not isinstance(ds.features["image"], HFImage):
        return ds.cast_column("image", HFImage())
    return ds


In [5]:
# Determine splits by inspecting the dataset dict
try:
    ds_all = load_dataset(HF_DATASET)
    ds_all = {k: ensure_image_column(v) for k, v in ds_all.items()}
    use_splits = list(ds_all.keys())
    print("Found splits:", use_splits)
except Exception as e:
    print("Failed to load dataset without split:", e)
    ds_all = None
    use_splits = ["train"]


Found splits: ['train', 'test']


In [6]:
manifest_rows = []
meta_rows = []

# Parallelized image extraction + metadata build

def process_example(
    ex,
    idx,
    split=None,
    images_dir=None,
    snapshot_dir=None,
    use_host_images=False,
    host_image_dir=None,
    read_image_sizes=False,
):
    img_id = ex.get("img_id") or safe_img_id(ex, split, idx)

    # Preferred path: use pre-downloaded host images
    if use_host_images:
        out_path = Path(host_image_dir) / f"{img_id}.jpg"
        if not out_path.exists():
            raise FileNotFoundError(
                f"Local host image missing for img_id={img_id}. "
                "Run host image download first (USE_HOST_IMAGES=1)."
            )
        orig_h = orig_w = None
        if read_image_sizes:
            try:
                with PILImage.open(out_path) as _im:
                    orig_w, orig_h = _im.size
            except Exception:
                orig_h = orig_w = None
        return {
            "_skip": False,
            "split": split,
            "img_id": img_id,
            "image_path": str(out_path),
            "exists": out_path.exists(),
            "orig_height": orig_h,
            "orig_width": orig_w,
            "question": ex.get("question"),
            "answer": ex.get("answer"),
            "question_type": ex.get("question_type"),
            "answer_type": ex.get("answer_type"),
            "question_class": ex.get("question_class"),
            "complexity": ex.get("complexity"),
            "original": ex.get("original"),
        }

    # Fallback path: use image field / local snapshot mapping
    img = ex.get("image")
    if isinstance(img, str):
        # If this is a HF URL and we have a local snapshot, load locally only.
        if snapshot_dir is not None:
            local_path = hf_url_to_local_path(img, Path(snapshot_dir))
            if local_path is None or not local_path.exists():
                raise FileNotFoundError(
                    f"Local snapshot missing image: {img}. "
                    "Re-run snapshot_download with FORCE_SNAPSHOT_DOWNLOAD=1 after rate limits reset."
                )
            img = PILImage.open(local_path).convert("RGB")
        else:
            try:
                img = PILImage.open(img).convert("RGB")
            except Exception:
                img = None

    if img is None:
        img_path_field = None
        for k in ["image_path", "img_path", "path", "file_name", "file_path"]:
            if k in ex and ex[k]:
                img_path_field = ex[k]
                break
        if img_path_field is None:
            return {
                "_skip": True,
                "split": split,
                "img_id": None,
                "image_path": None,
                "exists": False,
                "orig_height": None,
                "orig_width": None,
                "question": ex.get("question"),
                "answer": ex.get("answer"),
                "question_type": ex.get("question_type"),
                "answer_type": ex.get("answer_type"),
                "question_class": ex.get("question_class"),
                "complexity": ex.get("complexity"),
                "original": ex.get("original"),
            }
        try:
            img = PILImage.open(img_path_field).convert("RGB")
        except Exception:
            return {
                "_skip": True,
                "split": split,
                "img_id": None,
                "image_path": None,
                "exists": False,
                "orig_height": None,
                "orig_width": None,
                "question": ex.get("question"),
                "answer": ex.get("answer"),
                "question_type": ex.get("question_type"),
                "answer_type": ex.get("answer_type"),
                "question_class": ex.get("question_class"),
                "complexity": ex.get("complexity"),
                "original": ex.get("original"),
            }

    out_path = Path(images_dir) / split / f"{img_id}.jpg"
    if not out_path.exists():
        save_image(img, out_path)

    return {
        "_skip": False,
        "split": split,
        "img_id": img_id,
        "image_path": str(out_path),
        "exists": out_path.exists(),
        "orig_height": getattr(img, "height", None),
        "orig_width": getattr(img, "width", None),
        "question": ex.get("question"),
        "answer": ex.get("answer"),
        "question_type": ex.get("question_type"),
        "answer_type": ex.get("answer_type"),
        "question_class": ex.get("question_class"),
        "complexity": ex.get("complexity"),
        "original": ex.get("original"),
    }


# Ensure host images are present before mapping
if USE_HOST_IMAGES:
    _ensure_host_images()

# Now that snapshots/images are ready, go offline to avoid HF rate limits
if USE_LOCAL_SNAPSHOT:
    _maybe_set_offline()

for split in use_splits:
    print(f"Processing split: {split}")
    try:
        if ds_all is not None:
            ds = ds_all[split]
        else:
            ds = load_dataset(HF_DATASET, split=split)
        ds = ensure_image_column(ds)
    except Exception as e:
        print(f"Split {split} unavailable: {e}")
        continue
    print("  Split size:", len(ds))
    if len(ds) == 0:
        continue
    if MAX_SAMPLES_PER_SPLIT is not None:
        max_n = min(MAX_SAMPLES_PER_SPLIT, len(ds))
        ds = ds.select(range(max_n))
        print(f"  Truncated to {max_n} samples")

    num_workers = min(NUM_WORKERS, os.cpu_count() or NUM_WORKERS)
    ds_proc = ds.map(
        process_example,
        with_indices=True,
        num_proc=num_workers,
        fn_kwargs={
            "split": split,
            "images_dir": str(IMAGES_DIR),
            "snapshot_dir": str(LOCAL_SNAPSHOT_DIR) if USE_LOCAL_SNAPSHOT else None,
            "use_host_images": USE_HOST_IMAGES,
            "host_image_dir": str(HOST_IMAGE_DIR),
            "read_image_sizes": READ_IMAGE_SIZES,
        },
        load_from_cache_file=False,
        desc=f"Saving images/metadata ({split})",
    )

    if "_skip" in ds_proc.column_names:
        ds_proc = ds_proc.filter(lambda x: not x["_skip"], num_proc=num_workers)

    df = ds_proc.to_pandas()

    # Ensure required columns exist
    for col in ["question", "answer", "question_type", "answer_type", "question_class", "complexity", "original"]:
        if col not in df.columns:
            df[col] = None

    manifest_rows.extend(
        df[["split", "img_id", "image_path", "exists"]].to_dict("records")
    )
    meta_rows.extend(
        df[[
            "split",
            "img_id",
            "image_path",
            "question",
            "answer",
            "question_type",
            "answer_type",
            "question_class",
            "complexity",
            "original",
            "orig_height",
            "orig_width",
        ]].to_dict("records")
    )

manifest = pd.DataFrame(manifest_rows)
meta = pd.DataFrame(meta_rows)


Setting TOKENIZERS_PARALLELISM=false for forked processes.


Processing split: train
  Split size: 143594


Saving images/metadata (train) (num_proc=32):   0%|          | 0/143594 [00:00<?, ? examples/s]

Setting TOKENIZERS_PARALLELISM=false for forked processes.


Processing split: test
  Split size: 15955


Saving images/metadata (test) (num_proc=32):   0%|          | 0/15955 [00:00<?, ? examples/s]

In [7]:
# Preserve raw metadata before split assignment
meta_raw = meta.copy()

# Create train/val/test splits if dataset doesn't already include them
if 'split' not in meta.columns or meta['split'].nunique() <= 1:
    if not (0 < TRAIN_FRAC < 1 and 0 < VAL_FRAC < 1 and TRAIN_FRAC + VAL_FRAC < 1):
        raise ValueError('Check TRAIN_FRAC/VAL_FRAC: must sum to < 1')
    rng = random.Random(SPLIT_SEED)
    img_ids = sorted(meta['img_id'].dropna().unique())
    rng.shuffle(img_ids)
    n = len(img_ids)
    n_train = int(n * TRAIN_FRAC)
    n_val = int(n * VAL_FRAC)
    train_ids = set(img_ids[:n_train])
    val_ids = set(img_ids[n_train:n_train + n_val])

    def assign_split(img_id):
        if img_id in train_ids:
            return 'train'
        if img_id in val_ids:
            return 'validation'
        return 'test'

    if SPLIT_BY_IMAGE:
        meta['split'] = meta['img_id'].apply(assign_split)
    else:
        idx = list(range(len(meta)))
        rng.shuffle(idx)
        n_train_rows = int(len(idx) * TRAIN_FRAC)
        n_val_rows = int(len(idx) * VAL_FRAC)
        labels = (['train'] * n_train_rows +
                  ['validation'] * n_val_rows +
                  ['test'] * (len(idx) - n_train_rows - n_val_rows))
        meta['split'] = ''
        meta.loc[idx, 'split'] = labels

    # Update manifest to match new splits (image_path stays the same)
    split_map = meta.drop_duplicates('img_id').set_index('img_id')['split'].to_dict()
    manifest['split'] = manifest['img_id'].map(split_map)
    print('Assigned splits by img_id:', meta['split'].value_counts().to_dict())
else:
    print('Using existing splits:', meta['split'].value_counts().to_dict())


Using existing splits: {'train': 143594, 'test': 15955}


In [8]:
manifest.to_csv(IMAGE_MANIFEST_CSV, index=False)
meta_raw.to_csv(RAW_META_CSV, index=False)
meta.to_csv(ENRICHED_META_CSV, index=False)

print("Saved manifest:", IMAGE_MANIFEST_CSV, "rows:", len(manifest))
print("Saved metadata:", RAW_META_CSV, "rows:", len(meta))
if len(meta) > 0:
    print("Records per split:")
    print(meta.groupby("split").size())
else:
    print("No records written; check dataset/source settings.")


Saved manifest: out/manifests/image_manifest.csv rows: 159549
Saved metadata: out/metadata/metadata_raw.csv rows: 159549
Records per split:
split
test      15955
train    143594
dtype: int64


In [9]:
print("Nulls per column:")
print(meta.isnull().sum())


Nulls per column:
split                  0
img_id                 0
image_path             0
question               0
answer                 0
question_type     159549
answer_type       159549
question_class         0
complexity             0
original               0
orig_height       159549
orig_width        159549
dtype: int64
